# Simulation Survey

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import os
import json
import re
import random
import csv
import numpy as np
from typing import Literal, Dict, Any, List, Tuple
from pydantic import BaseModel, Field
from collections import defaultdict

In [ ]:
class Question(BaseModel):
    id: str
    prompt: str
    type: str


class ComparisonQuestion(Question):
    type: Literal['comparison'] = 'comparison'
    options: list[str] = Field(default_factory=lambda: ['left', 'equal', 'right'])  # noqa

In [ ]:
NUM_CONTESTANTS = 4

COVAL = 'Coval'
CEKURA = 'Cekura'
EVALION = 'Evalion'
APPLAUSE = 'Applause'
OLIVIA = 'Olivia'
CHLOE = 'Chloe'
SOPHIA = 'Sophia'
SCENARIO_ADHERENCE = 'Scenario Adherence'
HUMAN_NATURALNESS = 'Human Naturalness'
PERSONA_ADHERENCE = 'Persona Adherence'
DRAW = 'Draw'

# Survey Constants
STUDY_ID = 'study-pairwise-comparison'
SURVEY_PARTICIPANTS = 10

DATA_FOLDER = os.path.abspath('.')
FOLDER_SURVEYS = os.path.join(DATA_FOLDER, 'surveys')
FOLDER_RESPONSES = os.path.join(DATA_FOLDER, 'responses')
FOLDER_SURVEYS_APPLAUSE = os.path.join(DATA_FOLDER, 'surveys-applause')
FOLDER_RESPONSES_APPLAUSE = os.path.join(DATA_FOLDER, 'responses-applause')
RESULTS_FOLDER = os.path.join(DATA_FOLDER, 'results')
assert os.path.exists(FOLDER_SURVEYS)
assert os.path.exists(FOLDER_RESPONSES)
assert os.path.exists(FOLDER_SURVEYS_APPLAUSE)
assert os.path.exists(FOLDER_RESPONSES_APPLAUSE)
assert os.path.exists(RESULTS_FOLDER)
OPTIONS = [
    'Left recording',
    'Right recording',
    'No difference between recordings',
]

QUESTION_PROMPTS = {
    SCENARIO_ADHERENCE: {
        'Completeness': "Which audio is more aligned with the following statement?\nThe customer talked about all the issues in the scenario.",
        'Accuracy': "Which audio is more aligned with the following statement?\nThe customer explained their problems clearly and correctly.",
        'Goal Pursuit': "Which audio is more aligned with the following statement?\nThe customer made sure that all the problems they mentioned were addressed during the call (for example by asking follow-up questions, coming back to an unresolved step...)",
        'Hallucinations': 'Which audio is more aligned with the following statement?\nThe customer brought up problems that were not in the scenario.',
        'Overall Adherence': 'Overall in which audio did the customer act more according to the scenario?',
    },
    HUMAN_NATURALNESS: {
        'Voice Naturalness': "In which audio the customer's voice sounds more human, not robotic?",
        'Speaking Flow': 'In which audio the customer talks at a more natural pace -- smooth, not too fast, slow, or broken up?',
        'Tone and Emotion': "In which audio the customer's tone makes their feelings more clear (e.g., happy, annoyed, stressed, confused)?",
        'Word Choice': 'In which audio the customer uses more simple, natural words that are easy to understand?',
        'Response Fit': "In which audio the customer's replies make more sense and better follow the conversation?",
        'Overall Naturalness': 'Overall, in which audio the customer sounds more like a real person?',
    },
    PERSONA_ADHERENCE: {
        'Emotional Tone': "In which audio the customer sounds more calm?",
        'Cooperation': "In which audio the customer is more cooperative in trying to solve the problem with the support agent?",
        'Communication Style': "In which audio the customer explains themselves more clearly and concisely, without being overly wordy?",
        'Respect': "In which audio the customer uses more polite and respectful language towards the agent?",
        'Patience': "In which audio the customer sounds more patient and willing to wait, not rushed or demanding?",
    }
}

QUESTIONS = []
i = 0
for question_type, question_prompts in QUESTION_PROMPTS.items():
    for question_prompt in question_prompts.values():
        i += 1
        QUESTIONS.append(
            ComparisonQuestion(
                id=f'{STUDY_ID}-{i:03}',
                prompt=question_prompt,
                options=OPTIONS
            )
        )


METRIC_WEIGHTS = {
    SCENARIO_ADHERENCE: 0.4,
    HUMAN_NATURALNESS: 0.3,
    PERSONA_ADHERENCE: 0.3,
}
assert sum(METRIC_WEIGHTS.values()) == 1.0

from customer_support_scenarios import CUSTOMER_SUPPORT_TEST_CASES

print('Number of questions:', len(QUESTIONS))
print('Number of test cases:', len(CUSTOMER_SUPPORT_TEST_CASES))

In [ ]:
SURVEY_FILES = []
for file in os.listdir(FOLDER_SURVEYS):
    with open(os.path.join(FOLDER_SURVEYS, file), 'r') as f:
        data = json.load(f)
        SURVEY_FILES.append(data)
for file in os.listdir(FOLDER_SURVEYS_APPLAUSE):
    with open(os.path.join(FOLDER_SURVEYS_APPLAUSE, file), 'r') as f:
        data = json.load(f)
        SURVEY_FILES.append(data)
# Single ID per survey
assert len(set([sf['file_data']['uuid'] for sf in SURVEY_FILES])) == len(SURVEY_FILES)
# Two files with different IDs per survey
assert all([len(sf['recordings']) == 2 for sf in SURVEY_FILES])
assert all([sf['recordings'][0]['id'] != sf['recordings'][1]['id'] for sf in SURVEY_FILES])
# Single pair of recordings per survey
assert len(set([(survey_file['recordings'][0]['id'], survey_file['recordings'][1]['id']) for survey_file in SURVEY_FILES])) == len(SURVEY_FILES)
# 6 pairs * 15 scenarios * 3 personas = 270 surveys
assert len(SURVEY_FILES) == 270
print(f'{len(SURVEY_FILES)} surveys found')


surveys_per_simulation = defaultdict(int)
for survey_file in SURVEY_FILES:
    assert len(survey_file['recordings']) == 2
    surveys_per_simulation[survey_file['recordings'][0]['id']] += 1
    surveys_per_simulation[survey_file['recordings'][1]['id']] += 1
# 4 providers * 15 scenarios * 3 personas = 180 simulations
assert len(surveys_per_simulation) == 180
# Each simulation has exactly 2 surveys (one per each provider it is compared to)
assert all(v == NUM_CONTESTANTS - 1 for v in surveys_per_simulation.values())

In [ ]:
def validate_response_filename(filename):
    pattern = r"^([0-9a-fA-F-]{36})-([A-Za-z0-9]{24}).json$"
    match = re.match(pattern, filename)
    if match:
        return True
    else:
        return False


RESPONSES = []
for folder in [FOLDER_RESPONSES, FOLDER_RESPONSES_APPLAUSE]:
    for file in os.listdir(folder):
        filename = os.path.basename(file)
        if not validate_response_filename(filename):
            print(f"Skipping file {filename} because it has an invalid filename, might not be a response from Prolific")
            continue
        with open(os.path.join(folder, file), 'r') as f:
            response = json.load(f)
            if response['pageId'] != STUDY_ID:
                raise ValueError(f"File {filename} is not in the study {STUDY_ID}")
            # Find the associated survey file
            survey_file = None
            for sf in SURVEY_FILES:
                if sf['file_data']['uuid'] == response['uuid']:
                    if survey_file is not None:
                        raise ValueError(f"Response {filename} has a duplicate ID in survey files")
                    survey_file = sf
                    break
            if survey_file is None:
                raise ValueError(f"Response {filename} has an ID not found in survey files")
            # Ensure the file IDs match with the survey file
            # Note that, since the tool randomises the order, we need to check both possible orderings
            assert len(response['audioListeningAnalytics']['audioFiles']) == 2
            id_left = response['audioListeningAnalytics']['audioFiles'][0]['id']
            id_right = response['audioListeningAnalytics']['audioFiles'][1]['id']
            survey_recording1 = survey_file['recordings'][0]
            survey_recording2 = survey_file['recordings'][1]
            assert id_left != id_right
            assert survey_recording1['id'] != survey_recording2['id']
            assert id_left in [survey_recording1['id'], survey_recording2['id']]
            assert id_right in [survey_recording1['id'], survey_recording2['id']]
            if id_left == survey_recording1['id']:
                simulation_left = survey_recording1['source_id'].split('-')
                simulation_right = survey_recording2['source_id'].split('-')
            else:
                simulation_left = survey_recording2['source_id'].split('-')
                simulation_right = survey_recording1['source_id'].split('-')
            # Extract the test parameters and add them to the response
            contestant_left = simulation_left[0]
            contestant_right = simulation_right[0]
            test_left = simulation_right[1]
            test_right = simulation_right[1]
            persona_left = simulation_left[2]
            persona_right = simulation_right[2]
            assert contestant_left in [COVAL, CEKURA, EVALION, APPLAUSE]
            assert contestant_right in [COVAL, CEKURA, EVALION, APPLAUSE]
            assert contestant_left != contestant_right
            assert test_left == test_right
            assert persona_left == persona_right
            test_difficulty = None
            for test_case in CUSTOMER_SUPPORT_TEST_CASES:
                if test_case['name'] == test_left:
                    if test_difficulty is not None:
                        raise ValueError(f"Inconsistent test difficulty in response {filename}")
                    test_difficulty = test_case['difficulty']
            if test_difficulty is None:
                raise ValueError(f"Test difficulty not found for test {test_left}")
            response['test_name'] = test_left
            response['persona'] = persona_left
            response['test_difficulty'] = test_difficulty
            response['left_contestant'] = contestant_left
            response['right_contestant'] = contestant_right
            RESPONSES.append(response)
# Ensure each participant has one response per survey file
assert len(set([(r['uuid'], r['prolificId']) for r in RESPONSES])) == len(RESPONSES)
print(f"Found {len(RESPONSES)} responses")

# Ensure each survey has 10 participants
responses_per_survey = defaultdict(int)
for response in RESPONSES:
    responses_per_survey[response['uuid']] += 1
assert len(responses_per_survey) == 270
assert all(count == SURVEY_PARTICIPANTS for count in responses_per_survey.values())

In [ ]:
def compute_stats(responses: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    question_match = {}
    for question in QUESTIONS:
        for metric_key, submetrics in QUESTION_PROMPTS.items():
            for submetric_key, submetric_prompt in submetrics.items():
                if question.prompt == submetric_prompt:
                    if question.id in question_match:
                        raise ValueError(f"Question {question.id} has multiple prompts")
                    question_match[question.id] = {
                        'metric': metric_key,
                        'submetric': submetric_key,
                    }

    stats = []
    for response in responses:
        for question in response['responses']:
            assert question['response'] in OPTIONS
            if question['questionId'] not in question_match:
                raise ValueError(f"Question {question['questionId']} not found in question_match")
            metric_key, submetric_key = question_match[question['questionId']]['metric'], question_match[question['questionId']]['submetric']
            if question['response'] == 'Left recording':
                winner = response['left_contestant']
                loser = response['right_contestant']
            elif question['response'] == 'Right recording':
                winner = response['right_contestant']
                loser = response['left_contestant']
            elif question['response'] == 'No difference between recordings':
                winner = DRAW
                loser = DRAW
            else:
                raise ValueError(f"Question {question['questionId']} has an invalid response")
            # Process special cases
            if metric_key == SCENARIO_ADHERENCE and submetric_key == 'Hallucinations':
                # Scenario Adherence - Hallucinations is a special case where the winner is the loser
                winner, loser = loser, winner
            if metric_key == PERSONA_ADHERENCE:
                if submetric_key == 'Emotional Tone':
                    # Question: In which audio the customer sounds more calm?
                    if response['persona'] == OLIVIA:
                        winner, loser = loser, winner
                    elif response['persona'] == CHLOE:
                        winner, loser = winner, loser
                    elif response['persona'] == SOPHIA:
                        winner, loser = loser, winner
                    else:
                        raise ValueError(f"Question {question['questionId']} has an invalid persona")
                elif submetric_key == 'Cooperation':
                    # Question: In which audio the customer sounds more cooperative?
                    if response['persona'] == OLIVIA:
                        winner, loser = loser, winner
                    elif response['persona'] == CHLOE:
                        winner, loser = winner, loser
                    elif response['persona'] == SOPHIA:
                        winner, loser = winner, loser
                    else:
                        raise ValueError(f"Question {question['questionId']} has an invalid persona")
                elif submetric_key == 'Communication Style':
                    # Question: In which audio the customer explains themselves more clearly and concisely, without being overly wordy?
                    if response['persona'] == OLIVIA:
                        continue
                    elif response['persona'] == CHLOE:
                        winner, loser = loser, winner
                    elif response['persona'] == SOPHIA:
                        winner, loser = winner, loser
                    else:
                        raise ValueError(f"Question {question['questionId']} has an invalid persona")
                elif submetric_key == 'Respect':
                    # Question: In which audio the customer uses more polite and respectful language towards the agent?
                    if response['persona'] == OLIVIA:
                        winner, loser = loser, winner
                    elif response['persona'] == CHLOE:
                        winner, loser = winner, loser
                    elif response['persona'] == SOPHIA:
                        continue
                    else:
                        raise ValueError(f"Question {question['questionId']} has an invalid persona")
                elif submetric_key == 'Patience':
                    # Question: In which audio the customer sounds more patient and willing to wait, not rushed or demanding?
                    if response['persona'] == OLIVIA:
                        winner, loser = loser, winner
                    elif response['persona'] == CHLOE:
                        winner, loser = winner, loser
                    elif response['persona'] == SOPHIA:
                        winner, loser = loser, winner
                    else:
                        raise ValueError(f"Question {question['questionId']} has an invalid persona")
                else:
                    raise ValueError(f"Question {question['questionId']} has an invalid submetric")
            stats.append({
                'test_name': response['test_name'],
                'test_difficulty': response['test_difficulty'],
                'persona': response['persona'],
                'contestants': sorted([response['left_contestant'], response['right_contestant']]),
                'metric': metric_key,
                'submetric': submetric_key,
                'winner': winner,
                'loser': loser,
            })
    return stats


def compute_win_counts(stats: List[Dict[str, Any]], include_draws: bool = True) -> Dict[Tuple[str, str, str, str], int]:
    raw_scores = defaultdict(int)
    for r in stats:
        winner = r['winner']
        metric = r['metric']
        submetric = r['submetric']
        persona = r['persona']
        test_name = r['test_name']
        if winner == DRAW:
            if not include_draws:
                continue
            else:
                contestants = r['contestants']
                raw_scores[(test_name, persona, metric, submetric, contestants[0])] += 0.5
                raw_scores[(test_name, persona, metric, submetric, contestants[1])] += 0.5
        else:
            raw_scores[(test_name, persona, metric, submetric, winner)] += 1
    return raw_scores


def compute_elo(
    stats: List[Dict[str, Any]],
    k_factor: float = 32.0,
    initial_rating: float = 1500.0,
    include_draws: bool = True
) -> Dict[Tuple[str, str, str, str], float]:
    """
    Compute Elo ratings for contestants, tracked independently for each
    (test_name, persona, metric, submetric, contestant) tuple.

    Each match is a comparison between two contestants for a given test, persona, metric, and submetric.
    Elo ratings are updated for both contestants after each match, using the classic Elo formula.
    Draws are handled by adding two matches (one for each contestant as winner/loser) if include_draws is True.

    Args:
        stats: List of dicts, each representing a match with keys:
            - 'test_name': str
            - 'persona': str
            - 'contestants': list[str] of length 2
            - 'metric': str
            - 'submetric': str
            - 'winner': str (one of the contestants or 'Draw')
            - 'loser':  str (one of the contestants or 'Draw')
        k_factor: The Elo K-factor (step size for rating updates).
        initial_rating: The starting Elo rating for any unseen contestant in a bucket.
        include_draws: If True, draws are included as two half-matches; if False, draws are ignored.

    Returns:
        Dict mapping (test_name, persona, metric, submetric, contestant) to final Elo rating.
    """
    matches = []
    for stat in stats:
        if stat['winner'] == DRAW:
            assert stat['loser'] == DRAW
            if not include_draws:
                continue
            matches.append({
                'test_name': stat['test_name'],
                'persona': stat['persona'],
                'contestants': stat['contestants'],
                'metric': stat['metric'],
                'submetric': stat['submetric'],
                'winner': stat['contestants'][0],
                'loser': stat['contestants'][1]
            })
            matches.append({
                'test_name': stat['test_name'],
                'persona': stat['persona'],
                'contestants': stat['contestants'],
                'metric': stat['metric'],
                'submetric': stat['submetric'],
                'winner': stat['contestants'][1],
                'loser': stat['contestants'][0]
            })
        else:
            matches.append({
                'test_name': stat['test_name'],
                'persona': stat['persona'],
                'contestants': stat['contestants'],
                'metric': stat['metric'],
                'submetric': stat['submetric'],
                'winner': stat['winner'],
                'loser': stat['loser'],
            })
    
    # We shuffle the matches to ensure that the order of the matches does not affect the results
    random.seed(42)
    random.shuffle(matches)

    # Ratings keyed by (test_name, persona, metric, submetric, contestant)
    ratings: Dict[Tuple[str, str, str, str, str], float] = defaultdict(lambda: initial_rating)

    def expected_score(r_a: float, r_b: float) -> float:
        # Classic Elo expectation
        return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

    for m in matches:
        test_name  = m['test_name']
        persona    = m["persona"]
        metric     = m["metric"]
        submetric  = m["submetric"]
        a, b       = m["contestants"]
        winner     = m["winner"]
        loser      = m["loser"]

        key_a = (test_name, persona, metric, submetric, a)
        key_b = (test_name, persona, metric, submetric, b)

        ra = ratings[key_a]
        rb = ratings[key_b]

        if winner == DRAW or loser == DRAW:
            raise ValueError(f"Invalid match: {m}")

        # Determine outcomes
        if winner == a and loser == b:
            sa, sb = 1.0, 0.0
        elif winner == b and loser == a:
            sa, sb = 0.0, 1.0
        else:
            raise ValueError(f"Invalid match: {m}")

        ea = expected_score(ra, rb)
        eb = expected_score(rb, ra)

        # Update ratings
        ra_new = ra + k_factor * (sa - ea)
        rb_new = rb + k_factor * (sb - eb)

        ratings[key_a] = ra_new
        ratings[key_b] = rb_new

    return ratings


def score_arrays(scores, metric):
    assert metric in [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    submetrics = sorted(set([k[3] for k in scores.keys() if k[2] == metric]))
    assert submetrics == ['PCA'] or submetrics == sorted(list(QUESTION_PROMPTS[metric].keys()))
    PERSONAS = [OLIVIA, CHLOE, SOPHIA]
    CONTESTANTS = [COVAL, CEKURA, EVALION]
    TEST_NAMES = sorted(set([t[0] for t in scores.keys()]))
    metric_scores = defaultdict(list)
    for persona in PERSONAS:
        for test_name in TEST_NAMES:
            for contestant in CONTESTANTS:
                for submetric in submetrics:
                    score = scores[(test_name, persona, metric, submetric, contestant)]
                    value = (score, (test_name, persona, metric, contestant))
                    metric_scores[submetric].append(value)
    # Assert that the submetrics list are consistent
    combos = [x[1] for x in metric_scores[submetrics[0]]]
    for submetric in submetrics[1:]:
        assert [x[1] for x in metric_scores[submetric]] == combos
    return dict(metric_scores)


def first_pca_component(data_dict):
    # Convert dict of lists -> 2D numpy array
    # Shape: (n_samples, n_features)
    arr = np.array(list(data_dict.values())).T
    
    # Center the data (subtract mean per feature)
    X = arr - np.mean(arr, axis=0)
    
    # Compute covariance matrix
    cov = np.cov(X, rowvar=False)
    
    # Eigen decomposition
    eigvals, eigvecs = np.linalg.eigh(cov)
    
    # Get principal component with largest eigenvalue
    first_pc = eigvecs[:, np.argmax(eigvals)]
    
    # Project data onto the first PC
    scores = X @ first_pc

    # Flip the PC if it has a negative sum
    if np.sum(first_pc) < 0:
        first_pc = -first_pc
        scores = -scores
    
    return scores.tolist()


def compute_pca_scores(scores: Dict[Tuple[str, str, str, str, str], float]) -> Dict[Tuple[str, str, str, str, str], float]:
    METRICS = [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    result = {}
    for metric in METRICS:
        metric_scores = score_arrays(scores, metric)
        metric_array = {
            submetric: [s[0] for s in metric_scores[submetric]]
            for submetric in metric_scores.keys() if submetric not in ['Overall Adherence', 'Overall Naturalness']
        }
        pca_array = first_pca_component(metric_array)
        submetric = list(metric_scores.keys())[0]
        for score, pca_score in zip(metric_scores[submetric], pca_array):
            (test_name, persona, metric, contestant) = score[1]
            result[(test_name, persona, metric, 'PCA', contestant)] = pca_score
    return result


def normalise_scores(scores: Dict[Tuple[str, str, str, str, str], float]) -> Dict[Tuple[str, str, str, str, str], float]:
    METRICS = [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    min_values = []
    max_values = []
    for metric in METRICS:
        metric_scores = score_arrays(scores, metric)
        for submetric in metric_scores.keys():
            submetric_scores = metric_scores[submetric]
            min_values.append(min([s[0] for s in submetric_scores]))
            max_values.append(max([s[0] for s in submetric_scores]))
    min_score = min(min_values)
    max_score = max(max_values)
    result = {}
    for metric in METRICS:
        metric_scores = score_arrays(scores, metric)
        for submetric in metric_scores.keys():
            submetric_scores = metric_scores[submetric]
            for score in submetric_scores:
                (test_name, persona, metric, contestant) = score[1]
                result[(test_name, persona, metric, submetric, contestant)] = 100 * (score[0] - min_score) / (max_score - min_score)
    return result


def score_metrics_in_order(metrics, include_combined = True):
    metric_rank = []
    for metric in metrics:
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'{metric} Score League - {suffix}')
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'{metric} Score League - {suffix} - PCA')
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'{metric} Score Elo - {suffix}')
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'{metric} Score Elo - {suffix} - PCA')
    if include_combined:
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'Overall Score League - {suffix}')
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'Overall Score League - {suffix} - PCA')
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'Overall Score Elo - {suffix}')
        for suffix in ['WD', 'ND']:
            metric_rank.append(f'Overall Score Elo - {suffix} - PCA')
    return metric_rank


def reorder_keys(row: Dict[str, Any], keys: List[str]) -> Dict[str, Any]:
    missing = [k for k in keys if k not in row]
    if missing:
        raise ValueError(f"Missing keys: {missing}")
    extra = [k for k in row.keys() if k not in keys]
    final_order = extra + keys
    return {k: row[k] for k in final_order}


def stats_table(
    stats: List[Dict[str, Any]],
    key_factor: float = 32.0,
    initial_rating: float = 1500.0
) -> List[Dict[str, Any]]:
    """
    Returns a flat list of rows with the final Elo ratings,
    helpful for printing or turning into a DataFrame.
    """

    METRICS = [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    PERSONAS = sorted([OLIVIA, CHLOE, SOPHIA])
    CONTESTANTS = sorted([COVAL, CEKURA, EVALION])
    TEST_NAMES = sorted(set([t['test_name'] for t in stats]))

    # Aggregate winner counts
    raw_scores_wd = compute_win_counts(stats, include_draws=True)
    elo_scores_wd = compute_elo(stats, key_factor, initial_rating, include_draws=True)
    raw_scores_nd = compute_win_counts(stats, include_draws=False)
    elo_scores_nd = compute_elo(stats, key_factor, initial_rating, include_draws=False)
    pca_raw_scores_wd = compute_pca_scores(raw_scores_wd)
    pca_elo_scores_wd = compute_pca_scores(elo_scores_wd)
    pca_raw_scores_nd = compute_pca_scores(raw_scores_nd)
    pca_elo_scores_nd = compute_pca_scores(elo_scores_nd)

    raw_scores_normalised_wd = normalise_scores(raw_scores_wd)
    elo_scores_normalised_wd = normalise_scores(elo_scores_wd)
    pca_raw_scores_normalised_wd = normalise_scores(pca_raw_scores_wd)
    pca_elo_scores_normalised_wd = normalise_scores(pca_elo_scores_wd)
    raw_scores_normalised_nd = normalise_scores(raw_scores_nd)
    elo_scores_normalised_nd = normalise_scores(elo_scores_nd)
    pca_raw_scores_normalised_nd = normalise_scores(pca_raw_scores_nd)
    pca_elo_scores_normalised_nd = normalise_scores(pca_elo_scores_nd)

    metric_rank = score_metrics_in_order(METRICS)

    rows = []
    for test_name in TEST_NAMES:
        for persona in PERSONAS:
            for contestant in CONTESTANTS:
                row = {
                    'Test Name': test_name,
                    'Persona': persona,
                    'Contestant': contestant,
                }
                for include_draws in [True, False]:
                    if include_draws:
                        raw_scores = raw_scores_wd
                        elo_scores = elo_scores_wd
                        raw_scores_normalised = raw_scores_normalised_wd
                        elo_scores_normalised = elo_scores_normalised_wd
                        pca_raw_scores_normalised = pca_raw_scores_normalised_wd
                        pca_elo_scores_normalised = pca_elo_scores_normalised_wd
                        suffix = 'WD'
                    else:
                        raw_scores = raw_scores_nd
                        elo_scores = elo_scores_nd
                        raw_scores_normalised = raw_scores_normalised_nd
                        elo_scores_normalised = elo_scores_normalised_nd
                        pca_raw_scores_normalised = pca_raw_scores_normalised_nd
                        pca_elo_scores_normalised = pca_elo_scores_normalised_nd
                        suffix = 'ND'
                    for metric in METRICS:
                        for submetric in QUESTION_PROMPTS[metric].keys():
                            row[f'{metric} - {submetric} League - {suffix}'] = raw_scores[(test_name, persona, metric, submetric, contestant)]
                    for metric in METRICS:
                        for submetric in QUESTION_PROMPTS[metric].keys():
                            row[f'{metric} - {submetric} Elo - {suffix}'] = elo_scores[(test_name, persona, metric, submetric, contestant)]
                    for metric in METRICS:
                        if metric in [SCENARIO_ADHERENCE, HUMAN_NATURALNESS]:
                            if metric == SCENARIO_ADHERENCE:
                                submetric = 'Overall Adherence'
                            else:
                                submetric = 'Overall Naturalness'
                            row[f'{metric} Score League - {suffix}'] = raw_scores_normalised[(test_name, persona, metric, submetric, contestant)]
                            row[f'{metric} Score League - {suffix} - PCA'] = pca_raw_scores_normalised[(test_name, persona, metric, 'PCA', contestant)]
                        elif metric == PERSONA_ADHERENCE:
                            row[f'{metric} Score League - {suffix}'] = pca_raw_scores_normalised[(test_name, persona, metric, 'PCA', contestant)]
                            row[f'{metric} Score League - {suffix} - PCA'] = pca_raw_scores_normalised[(test_name, persona, metric, 'PCA', contestant)]
                    for metric in METRICS:
                        if metric in [SCENARIO_ADHERENCE, HUMAN_NATURALNESS]:
                            if metric == SCENARIO_ADHERENCE:
                                submetric = 'Overall Adherence'
                            else:
                                submetric = 'Overall Naturalness'
                            row[f'{metric} Score Elo - {suffix}'] = elo_scores_normalised[(test_name, persona, metric, submetric, contestant)]
                            row[f'{metric} Score Elo - {suffix} - PCA'] = pca_elo_scores_normalised[(test_name, persona, metric, 'PCA', contestant)]
                        elif metric == PERSONA_ADHERENCE:
                            row[f'{metric} Score Elo - {suffix}'] = pca_elo_scores_normalised[(test_name, persona, metric, 'PCA', contestant)]
                            row[f'{metric} Score Elo - {suffix} - PCA'] = pca_elo_scores_normalised[(test_name, persona, metric, 'PCA', contestant)]
                    row[f'Overall Score League - {suffix}'] = 0
                    row[f'Overall Score League - {suffix} - PCA'] = 0
                    row[f'Overall Score Elo - {suffix}'] = 0
                    row[f'Overall Score Elo - {suffix} - PCA'] = 0
                    for metric in METRICS:
                        row[f'Overall Score League - {suffix}'] += row[f'{metric} Score League - {suffix}'] * METRIC_WEIGHTS[metric]
                        row[f'Overall Score League - {suffix} - PCA'] += row[f'{metric} Score League - {suffix} - PCA'] * METRIC_WEIGHTS[metric]
                        row[f'Overall Score Elo - {suffix}'] += row[f'{metric} Score Elo - {suffix}'] * METRIC_WEIGHTS[metric]
                        row[f'Overall Score Elo - {suffix} - PCA'] += row[f'{metric} Score Elo - {suffix} - PCA'] * METRIC_WEIGHTS[metric]
                rows.append(reorder_keys(row, metric_rank))
    
    metric_rank_values = {}
    for row in rows:
        (test_name, persona, contestant) = (row['Test Name'], row['Persona'], row['Contestant'])
        for metric_rank_item in metric_rank:
            assert (test_name, persona, contestant, metric_rank_item) not in metric_rank_values
            metric_rank_values[(test_name, persona, contestant, metric_rank_item)] = (row[metric_rank_item], 0)

    for test_name in TEST_NAMES:
        for persona in PERSONAS:
            for contestant in CONTESTANTS:
                for metric_rank_item in metric_rank:
                    curr_value = metric_rank_values[(test_name, persona, contestant, metric_rank_item)]
                    if curr_value[1] != 0:
                        raise ValueError(f"Metric rank value for {test_name} {persona} {contestant} {metric_rank_item} is already set")
                    possible_values = sorted([
                        metric_rank_values[(test_name, persona, c, metric_rank_item)][0]
                        for c in CONTESTANTS
                    ], reverse=True)
                    curr_rank = possible_values.index(curr_value[0]) + 1
                    metric_rank_values[(test_name, persona, contestant, metric_rank_item)] = (curr_value[0], curr_rank)
    for row in rows:
        for metric_rank_item in metric_rank:
            row[metric_rank_item + ' Rank'] = metric_rank_values[(row['Test Name'], row['Persona'], row['Contestant'], metric_rank_item)][1]
    return rows


def score_fields_selected():
    submetric_fields = [
        'Scenario Adherence - Completeness League - WD',
        'Scenario Adherence - Accuracy League - WD',
        'Scenario Adherence - Goal Pursuit League - WD',
        'Scenario Adherence - Hallucinations League - WD',
        'Scenario Adherence - Overall Adherence League - WD',
        'Human Naturalness - Voice Naturalness League - WD',
        'Human Naturalness - Speaking Flow League - WD',
        'Human Naturalness - Tone and Emotion League - WD',
        'Human Naturalness - Word Choice League - WD',
        'Human Naturalness - Response Fit League - WD',
        'Human Naturalness - Overall Naturalness League - WD',
        'Persona Adherence - Emotional Tone League - WD',
        'Persona Adherence - Cooperation League - WD',
        'Persona Adherence - Communication Style League - WD',
        'Persona Adherence - Respect League - WD',
        'Persona Adherence - Patience League - WD',
    ]
    metric_fields = [
        'Scenario Adherence Score League - WD',
        'Human Naturalness Score League - WD',
        'Persona Adherence Score League - WD',
        'Overall Score League - WD',
    ]
    return submetric_fields, metric_fields


def summarise_scores(scores: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    fields = [
        'Test Name',
        'Difficulty',
        'Persona',
        'Contestant',
        'Is Winner',
    ]
    submetric_fields, metric_fields = score_fields_selected()
    extra_fields = submetric_fields + metric_fields + [f'{ef} Rank' for ef in metric_fields]
    scores_summarised = []
    for score in scores:
        row = {
            field.replace(' - WD', ''): '' if field in ['Is Winner', 'Difficulty'] else score[field] for field in fields + extra_fields
        }
        if score['Overall Score League - WD Rank'] == 1:
            row['Is Winner'] = 'Yes'
        else:
            row['Is Winner'] = 'No'
        for test_case in CUSTOMER_SUPPORT_TEST_CASES:
            if test_case['name'] == score['Test Name']:
                row['Difficulty'] = test_case['difficulty']
                break
        scores_summarised.append(row)
    return sorted(scores_summarised, key=lambda x: (x['Test Name'], x['Persona'], x['Contestant']))

In [ ]:
STATS = compute_stats(RESPONSES)
TEST_SCORES_ALL_METRICS = stats_table(STATS)
TEST_SCORES = summarise_scores(TEST_SCORES_ALL_METRICS)

In [ ]:
fields = TEST_SCORES[0].keys()
csv_path = os.path.join(RESULTS_FOLDER, "test_scores.csv")
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    writer.writeheader()
    writer.writerows(TEST_SCORES)


fields = TEST_SCORES_ALL_METRICS[0].keys()
csv_path = os.path.join(RESULTS_FOLDER, "test_scores_all_metrics.csv")
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    writer.writeheader()
    writer.writerows(TEST_SCORES_ALL_METRICS)

# Charts

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

In [ ]:
def plot_winner_counts(records, *, dpi=180, base_width=18, height_per_test=3.2, font_scale=1.0):
    """
    Cleaner, roomier plot of winner counts per (metric → test x persona), ignoring draws.

    Tunables
    --------
    dpi : int                High DPI for crisp text.
    base_width : float       Overall figure width (inches) for the 3 metric columns.
    height_per_test : float  Height (inches) added per unique test row.
    font_scale : float       Multiplier for all font sizes (e.g., 1.2 to enlarge).

    Returns
    -------
    matplotlib.figure.Figure
    """
    METRICS = [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    PERSONAS = [OLIVIA, CHLOE, SOPHIA]
    CONTESTANTS = [COVAL, CEKURA, EVALION, APPLAUSE] + [DRAW]

    # Aggregate winner counts, ignoring draws
    counts = defaultdict(int)
    for r in records:
        winner = r['winner']
        metric  = r['metric']
        persona = r['persona']
        test    = r['test_name'] + ' - ' + r['test_difficulty']
        assert metric in METRICS
        assert persona in PERSONAS
        assert winner in CONTESTANTS
        if metric == SCENARIO_ADHERENCE:
            if r['submetric'] != 'Overall Adherence':
                continue
        if metric == HUMAN_NATURALNESS:
            if r['submetric'] != 'Overall Naturalness':
                continue
        counts[(metric, test, persona, winner)] += 1

    test_names = set()
    for k in counts.keys():
        test_names.add(k[1])
    difficulty_order = {
        'Easy': 0,
        'Medium': 1,
        'Hard': 2,
    }
    test_names = sorted(test_names, key=lambda x: (int(difficulty_order[x.split(' - ')[1]]), x.split(' - ')[0]))

    # Sizing & fonts
    fig_height = max(3.0, height_per_test * len(test_names))
    fs_title   = 18 * font_scale
    fs_axis    = 12 * font_scale
    fs_small   = 10 * font_scale

    # Figure & outer layout
    fig = plt.figure(figsize=(base_width, fig_height), dpi=dpi, constrained_layout=False)
    # Leave more horizontal space between the metric blocks
    outer = fig.add_gridspec(nrows=1, ncols=3, wspace=0.15)  # increased from 0.03 to 0.18

    # Column headers
    for m_idx, metric in enumerate(METRICS):
        # Increase hspace for more vertical breathing room between subplots
        sub = outer[m_idx].subgridspec(
            nrows=len(test_names), ncols=len(PERSONAS), wspace=0.1, hspace=0.5  # was 0.18
        )

        # Add a bold column title centered above this metric column
        bbox = outer[m_idx].get_position(fig)
        x_center = (bbox.x0 + bbox.x1) / 2
        y_top = bbox.y1 + 0.015  # small offset above the top of the column
        fig.text(x_center, y_top, metric, ha='center', va='bottom',
                 fontsize=fs_title, weight='bold', transform=fig.transFigure)

        for t_idx, test in enumerate(test_names):
            # Determine a shared y-limit across personas for this (metric, test) row
            row_vals = []
            for persona in PERSONAS:
                row_vals.extend(counts.get((metric, test, persona, c), 0) for c in CONTESTANTS)
            y_max = max(row_vals) if row_vals else 0
            y_lim = max(1, y_max)  # show at least [0,1]

            for p_idx, persona in enumerate(PERSONAS):
                ax = fig.add_subplot(sub[t_idx, p_idx])

                vals = [counts.get((metric, test, persona, c), 0) for c in CONTESTANTS]
                ax.bar(range(len(CONTESTANTS)), vals)

                ax.set_ylim(0, y_lim)
                ax.set_xlabel("")  # less clutter

                # Show persona across the very top row of each metric column
                if t_idx == 0:
                    ax.set_title(persona, fontsize=fs_axis, pad=6)

                # Set test_name as y-axis label only for the first subplot of the first metric
                if m_idx == 0 and p_idx == 0:
                    ax.set_ylabel(f'{test}', fontsize=fs_axis)
                    ax.tick_params(axis='y', labelsize=fs_small)
                else:
                    ax.set_ylabel("")
                    if p_idx == 0:
                        ax.tick_params(axis='y', labelsize=fs_small)
                    else:
                        ax.tick_params(axis='y', left=False, labelleft=False)

                # Show x tick labels in all subplots (not just the bottom row)
                ax.set_xticks(range(len(CONTESTANTS)))
                ax.set_xticklabels(CONTESTANTS, fontsize=fs_small, rotation=30, ha='right')

                # Subtle grid
                ax.yaxis.grid(True, linestyle=':', linewidth=0.6)
                ax.set_axisbelow(True)

    plt.show()

plot_winner_counts(STATS)

In [ ]:
def regression_analysis(stats, metric, mechanism='league', include_draws=True):
    assert metric in [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    submetrics = list(QUESTION_PROMPTS[metric].keys())
    if metric == SCENARIO_ADHERENCE:
        overall_metric = 'Overall Adherence'
    elif metric == HUMAN_NATURALNESS:
        overall_metric = 'Overall Naturalness'
    elif metric == PERSONA_ADHERENCE:
        overall_metric = None

    if mechanism == 'elo':
        scores = compute_elo(stats, include_draws=include_draws)
    elif mechanism == 'league':
        scores = compute_win_counts(stats, include_draws=include_draws)
    else:
        raise ValueError(f"Invalid mechanism: {mechanism}")
    metric_scores = score_arrays(scores, metric)

    metric_scores_values = {
        var: [x[0] for x in metric_scores[var]]
        for var in submetrics
    }

    if overall_metric is not None:
        submetrics = [overall_metric] + [s for s in submetrics if s != overall_metric]
    for submetric in submetrics:
        if submetric is None:
            continue
        submetrics_to_use = [s for s in submetrics if s not in [submetric, overall_metric]]
        # Now do an OLS regression of the overall metric on the submetrics
        # Stack predictors into a single 2D array
        X = np.column_stack([np.array(metric_scores_values[subm]) for subm in submetrics_to_use])
        # Add a column of ones for the intercept
        X = sm.add_constant(X)
        # Define target
        y = np.array(metric_scores_values[submetric])
        variable_names = ['Intercept'] + submetrics_to_use
        # Fit OLS regression
        model = sm.OLS(y, X).fit()
        model.model.data.xnames = variable_names  # rename variables
        # Print regression results
        if submetric == overall_metric:
            print(f"Full OLS Regression results for {submetric}:")
            print(model.summary())
        else:
            header = f"{'Variable':<20} {'Coef':>10} {'p-value':>12}"
            print('')
            print(f"Regression results for {submetric}: R2 = {model.rsquared:.4f}")
            print('')
            print(header)
            print('-' * len(header))
            for var, coef, pval in zip(model.model.data.xnames, model.params, model.pvalues):
                print(f"{var:<20} {coef:>10.4f} {pval:>12.4g}")

In [ ]:
regression_analysis(STATS, SCENARIO_ADHERENCE)

In [ ]:
regression_analysis(STATS, HUMAN_NATURALNESS)

In [ ]:
def analyze_persona_adherence_correlation(stats, mechanism='league', include_draws=True):
    if mechanism == 'elo':
        scores = compute_elo(stats, include_draws=include_draws)
    elif mechanism == 'league':
        scores = compute_win_counts(stats, include_draws=include_draws)
    else:
        raise ValueError(f"Invalid mechanism: {mechanism}")

    metric_scores = score_arrays(scores, PERSONA_ADHERENCE)
    var_names = list(QUESTION_PROMPTS[PERSONA_ADHERENCE].keys())
    metric_scores_values = {var: [x[0] for x in metric_scores[var]] for var in var_names}

    # Extract variable names and data
    X = np.array([metric_scores_values[var] for var in var_names]).T  # shape: (n_samples, n_variables)

    # --- Correlation Matrix ---
    corr_matrix = np.corrcoef(X, rowvar=False)

    # Plot correlation heatmap with annotations
    fig, ax = plt.subplots(figsize=(8, 6))
    cax = ax.matshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(cax)

    ax.set_xticks(range(len(var_names)))
    ax.set_yticks(range(len(var_names)))
    ax.set_xticklabels(var_names, rotation=45, ha="left")
    ax.set_yticklabels(var_names)
    ax.set_title("Correlation Matrix", pad=20)

    # Add correlation values in each cell (rounded to 2 decimals)
    n = len(var_names)
    for i in range(n):
        for j in range(n):
            val = corr_matrix[i, j]
            # Choose text color for readability
            text_color = "white" if abs(val) > 0.5 else "black"
            ax.text(j, i, f"{val:.2f}", va="center", ha="center", color=text_color)

    plt.tight_layout()
    plt.show()

In [ ]:
analyze_persona_adherence_correlation(STATS)

In [ ]:
def plot_metric_correlations_numpy(records: List[Dict[str, float]], metric: str = 'all') -> None:
    """
    Plot a correlation matrix heatmap for a list of metric dicts using only NumPy + matplotlib.
    - Expects each dict to have (some subset of) the same keys with float values in [0, 100].
    - Abbreviates names by taking the first letter of each word before ' - ',
      then keeping the trailing ' - ...' suffix parts.
      Examples:
        'Overall Score Elo - WD - PCA' -> 'OSE - WD - PCA'
        'Scenario Adherence Score League - ND' -> 'SASL - ND'
    """
    if not records:
        raise ValueError("records is empty")
    if metric == 'all':
        METRICS = [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    elif metric == 'combined':
        METRICS = []
    else:
        METRICS = [metric]

    ordered_keys = score_metrics_in_order(METRICS, include_combined = metric in ['all', 'combined'])

    # -------- Build data matrix (N samples x M features), fill with NaN --------
    N, M = len(records), len(ordered_keys)
    X = np.full((N, M), np.nan, dtype=float)
    for i, rec in enumerate(records):
        for j, key in enumerate(ordered_keys):
            if key in rec:
                try:
                    X[i, j] = float(rec[key])
                except (TypeError, ValueError):
                    X[i, j] = np.nan  # non-numeric -> NaN

    # -------- Pairwise Pearson correlation, ignoring NaNs pairwise --------
    def pairwise_corr(A: np.ndarray) -> np.ndarray:
        m = A.shape[1]
        C = np.full((m, m), np.nan, dtype=float)
        for i in range(m):
            xi = A[:, i]
            for j in range(m):
                xj = A[:, j]
                mask = (~np.isnan(xi)) & (~np.isnan(xj))
                n = int(mask.sum())
                if n >= 2:
                    a = xi[mask]
                    b = xj[mask]
                    a_center = a - a.mean()
                    b_center = b - b.mean()
                    denom = np.sqrt((a_center ** 2).sum()) * np.sqrt((b_center ** 2).sum())
                    if denom > 0:
                        C[i, j] = float((a_center @ b_center) / denom)
                    else:
                        C[i, j] = np.nan
        return C

    corr = pairwise_corr(X)

    # -------- Abbreviations --------
    def abbreviate(name: str) -> str:
        parts = [p.strip() for p in name.split(" - ")]
        left = parts[0] if parts else name
        left_abbr = "".join(w[0].upper() for w in left.split() if w)
        suffix = (" - " + " - ".join(parts[1:])) if len(parts) > 1 else ""
        return f"{left_abbr}{suffix}"

    labels = [abbreviate(k) for k in ordered_keys]

    # -------- Plot heatmap with annotations --------
    n = corr.shape[0]
    fig_size = max(8.0, n * 0.5)  # scale with number of variables
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    im = ax.imshow(corr, vmin=-1, vmax=1)  # default colormap
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Pearson r", rotation=90, va="center")

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticklabels(labels)

    # Annotate each cell (rounded to 2 decimals); blank for NaN
    for i in range(n):
        for j in range(n):
            val = corr[i, j]
            txt = "" if np.isnan(val) else f"{val:.2f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8)

    ax.set_title("Metric Correlation Heatmap")
    ax.set_xlabel("Metrics")
    ax.set_ylabel("Metrics")
    fig.tight_layout()
    plt.show()


def plot_metric_match_heatmap_numpy(records: List[Dict[str, int]], metric: str = 'all') -> None:
    """
    Given a list of dicts whose keys are the metric names and values are integers in {1,2,3},
    compute a pairwise matrix where each cell (i,j) is the percentage of entries for which
    the values of key_i and key_j match exactly (considering only rows where both are present).
    Plot this as a heatmap with percentages annotated in each cell (0–100%).

    No pandas used; only numpy + matplotlib.
    """
    if not records:
        raise ValueError("records is empty")
    
    if metric == 'all':
        METRICS = [SCENARIO_ADHERENCE, HUMAN_NATURALNESS, PERSONA_ADHERENCE]
    elif metric == 'combined':
        METRICS = []
    else:
        METRICS = [metric]

    ordered_keys = score_metrics_in_order(METRICS, include_combined = metric in ['all', 'combined'])
    ordered_keys = [ok + ' Rank' for ok in ordered_keys]

    # ---- Build data matrix (N samples x M features), fill with NaN ----
    N, M = len(records), len(ordered_keys)
    X = np.full((N, M), np.nan, dtype=float)
    for i, rec in enumerate(records):
        for j, key in enumerate(ordered_keys):
            if key in rec:
                try:
                    # allow ints 1..3; coerce others to NaN
                    val = float(rec[key])
                    if np.isfinite(val):
                        X[i, j] = val
                except (TypeError, ValueError):
                    pass  # leave as NaN

    # ---- Pairwise exact-match percentage, ignoring rows where either is NaN ----
    P = np.full((M, M), np.nan, dtype=float)
    for i in range(M):
        xi = X[:, i]
        for j in range(M):
            xj = X[:, j]
            mask = (~np.isnan(xi)) & (~np.isnan(xj))
            denom = int(mask.sum())
            if denom > 0:
                matches = int((xi[mask] == xj[mask]).sum())
                P[i, j] = 100.0 * matches / denom  # percentage 0..100

    # ---- Abbreviate labels: collapse first part to initials, keep suffix " - ..." ----
    def abbreviate(name: str) -> str:
        parts = [p.strip() for p in name.split(" - ")]
        left = parts[0] if parts else name
        left_abbr = "".join(w[0].upper() for w in left.split() if w)
        suffix = (" - " + " - ".join(parts[1:])) if len(parts) > 1 else ""
        return f"{left_abbr}{suffix}"

    labels = [abbreviate(k) for k in ordered_keys]

    # ---- Plot heatmap ----
    n = P.shape[0]
    if n == 0:
        raise ValueError("No features to plot.")
    fig_size = max(8.0, n * 0.5)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    im = ax.imshow(P, vmin=0, vmax=100)  # default colormap
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Exact match (%)", rotation=90, va="center")

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticklabels(labels)

    # Annotate each cell with percentage (blank if NaN)
    for i in range(n):
        for j in range(n):
            val = P[i, j]
            txt = "" if np.isnan(val) else f"{val:.0f}%"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8)

    ax.set_title("Pairwise Exact-Match Percentage Heatmap")
    ax.set_xlabel("Metrics")
    ax.set_ylabel("Metrics")
    fig.tight_layout()
    plt.show()

In [ ]:
plot_metric_correlations_numpy(TEST_SCORES_ALL_METRICS, 'combined')